# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
# Role
You are the EcoHome Energy Advisor, an expert AI assistant for smart-home customers who own
solar panels, electric vehicles and smart devices. You make DATA-DRIVEN decisions that reduce
electricity cost and environmental impact - you do not give generic advice.

# What you must do (follow these steps)
1. Understand the question: identify the device(s), the time frame, and the goal (cost, solar use, comfort).
2. Gather data with tools BEFORE answering. Never guess prices, weather or usage:
   - get_electricity_prices  -> hourly time-of-use prices, cheapest / peak hours, price spikes
   - get_weather_forecast    -> temperature, conditions, expected solar generation
   - query_energy_usage / get_recent_energy_summary -> the customer's historical consumption
   - query_solar_generation  -> historical solar production
   - search_energy_tips      -> best practices from the knowledge base (always use for "tips" or "how to")
   - calculate_energy_savings -> exact savings numbers (never do mental math for savings)
3. Analyse: combine price + solar forecast + usage history; pick the best hour(s) or setting.
4. Recommend: give a specific, actionable answer.
5. Quantify: provide estimated savings ($ per week/month/year) and CO2 impact where relevant.

# Key capabilities
- EV charging, HVAC/thermostat, appliance and pool-pump scheduling, solar maximisation, battery storage
- Multi-device optimisation, cost and ROI calculations, weather- and price-aware planning
- Use today's date from the context to resolve words like "tomorrow", "Wednesday" or "this week"
  (pass dates as YYYY-MM-DD).

# Recommendation format
- Start with a one-sentence direct answer (exact time window / temperature / action).
- "Why": cite the data you retrieved (prices in $/kWh, forecast, usage figures).
- "Savings": estimated dollars and CO2, showing the assumption (kWh, rates) you used.
- "Tips": cite knowledge-base advice by naming the source document, e.g. (source: tip_hvac_optimization.txt).
- Keep it concise and friendly; use bullets. Prefer solar-hour or off-peak scheduling, and avoid peak prices.
- Never compromise safety or basic comfort (e.g. keep temperatures reasonable).

# Tool usage rules
- query_energy_usage / query_solar_generation hold PAST data only. For tomorrow or the coming days use
  get_weather_forecast and get_electricity_prices instead of querying the database.
- Call each tool once per question unless you need a different date range; do not repeat identical calls.

# Error handling
- If a tool returns an error or no data, say so, use the other data you have, state your assumptions,
  and never invent numbers.
- If the question is not about home energy, politely redirect to energy topics.
- If a tool returns an error, retry at most ONCE with corrected arguments. If it still fails, say clearly
  which data was unavailable, use the data you do have, label every assumption, and never invent numbers.
- If key information is missing (e.g. kWh per run), state a reasonable assumption and proceed.

# Example questions you handle
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"
- "Suggest three ways I can reduce energy use based on my usage history."
- "How much can I save by running my dishwasher during off-peak hours?"
- "What's the best time to run my pool pump this week based on the weather forecast?"
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

Charge your electric car tomorrow (October 31) between 12:00 AM and 5:00 AM to minimize costs and maximize solar power usage.

**Why:**
- **Electricity Prices:** The rates during this time are $0.12/kWh, which is the lowest (off-peak).
- **Solar Generation:** The best solar hours are expected from 10:00 AM to 3:00 PM, with peak generation around noon. Charging overnight allows you to use solar power during the day.

**Savings:**
- Assuming you charge your EV with 30 kWh, charging during off-peak hours instead of peak hours (which can go up to $0.336/kWh) could save you approximately:
  - **Cost Savings:** $30 kWh * ($0.336 - $0.12) = $6.48
  - **CO2 Impact:** Charging during off-peak hours typically uses cleaner energy, reducing your carbon footprint.

**Tips:**
- Consider scheduling your EV charging to start just after midnight to take full advantage of the lower rates (source: tip_ev_charging_optimization.txt). 

Feel free to ask if you need more information!


In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [ ]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [7]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration"
    },
    {
        "id": "thermostat_1",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should recommend a specific temperature setting, mention the Wednesday price spike with rates, suggest pre-cooling or pre-heating before the spike, and cite HVAC tips from the knowledge base"
    },
    {
        "id": "usage_reduction_1",
        "question": "Suggest three ways I can reduce energy use based on my usage history.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should list three specific recommendations grounded in the customer's actual usage data (top consuming devices, peak hours) and supported by knowledge base tips"
    },
    {
        "id": "appliance_savings_1",
        "question": "How much can I save by running my dishwasher during off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should compare peak and off-peak rates, state an off-peak time window, and give quantified savings in dollars (per cycle and per year) with assumptions"
    },
    {
        "id": "pool_pump_1",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should recommend run times for the pool pump, consider sunny versus cloudy days from the forecast, factor in electricity rates, and mention best practices such as daytime solar hours"
    },
    {
        "id": "solar_maximization_1",
        "question": "How can I maximize my use of solar power over the next three days?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation"],
        "expected_response": "The response should use the 3-day forecast and historical solar production to identify the best solar hours per day and suggest shifting flexible loads (EV, laundry, pool pump) into those windows"
    },
    {
        "id": "cost_savings_1",
        "question": "How much money would I save per year by moving my EV charging from peak to off-peak hours? I charge 4 times a week, about 20 kWh each time.",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should state peak and off-peak rates, show the calculation using 80 kWh per week, and give weekly or annual savings in dollars"
    },
    {
        "id": "multi_device_1",
        "question": "I need to run my dishwasher, dryer and charge my EV tomorrow. What schedule do you recommend?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "The response should give a time slot for each of the three devices, avoid peak hours, prioritize solar hours when the forecast is good, and avoid overloading a single hour"
    },
    {
        "id": "battery_storage_1",
        "question": "How should I use a home battery to take advantage of tomorrow's electricity prices and weather?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should recommend when to charge (off-peak or solar hours) and when to discharge (peak hours), reference the forecast for solar refill, and cite energy storage best practices"
    },
    {
        "id": "usage_analysis_1",
        "question": "Which of my devices used the most energy and money in the last 7 days, and what should I change?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should rank devices by kWh and cost using real data from the database, identify the biggest opportunity, and give actionable changes with estimated savings"
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [8]:
CONTEXT = "Location: San Francisco, CA"

In [9]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------

Test 3: usage_reduction_1
Question: Suggest three ways I can reduce energy use based on my usage history.
--------------------------------------------------

Test 4: appliance_savings_1
Question: How much can I save by running my dishwasher during off-peak hours?
--------------------------------------------------

Test 5: pool_pump_1
Question: What's the best time to run my pool pump this week based on the weather forecast?
--------------------------------------------------

Test 6: solar_maximization_1
Question: How can I maximize my use of solar power over the next three days?
---------------------------------

In [10]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='8ceb3d0b-d6f9-486d-9bb0-9ba98bdda030'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_pUWAm1E68K5zAjnIDRfSxjfA', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_LFz9y5Uduvltp85B47ZB4QrM', 'function': {'arguments': '{"date": "2026-10-31"}', 'name': 'get_electricity_prices'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1592, 'total_tokens': 1653, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [17]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response.
    Metrics: ACCURACY, RELEVANCE, COMPLETENESS, USEFULNESS (0-100 each) + feedback for each."""
    import re
 
    text = final_response or ""
    low, q = text.lower(), question.lower()
 
    has_times = bool(re.search(r"\d{1,2}:\d{2}|\b\d{1,2}\s*(am|pm)\b", low))
    has_costs = bool(re.search(r"\$\s?\d+\.?\d*", text))
    has_temps = bool(re.search(r"\d+\s*°\s*[fc]|\d+\s*degrees", low))
    has_kwh = bool(re.search(r"\d+\.?\d*\s*kwh", low))
 
    def grade(s):
        return "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 70 else "D" if s >= 60 else "F"
 
    result = {}
 
    # ---------------- ACCURACY ----------------
    score, fb = 0, []
    if has_times or has_costs or has_temps or has_kwh:
        score += 35; fb.append("✓ Contains specific values (times, costs, temperatures or kWh)")
    else:
        fb.append("✗ No specific values - claims are not backed by numbers")
    if any(w in low for w in ["forecast", "weather", "price", "pricing", "rate", "usage", "consumption", "irradiance"]):
        score += 35; fb.append("✓ References retrieved data (prices / weather / usage)")
    else:
        fb.append("✗ Does not cite the data it is based on")
    if re.search(r"[x×*]\s*\$?\d|=\s*\**\$?\d|because|since|due to", low):
        score += 30; fb.append("✓ Shows calculations or reasoning")
    else:
        fb.append("✗ No calculation or reasoning shown")
    result["accuracy"] = {"score": score, "feedback": " | ".join(fb)}
 
    # ---------------- RELEVANCE ----------------
    topics = {
        "EV": r"\b(ev|electric (car|vehicle))\b", "charging": r"\bcharg", "thermostat/HVAC": r"thermostat|\bhvac\b|temperature",
        "dishwasher": r"dishwasher", "solar": r"\bsolar\b", "pool pump": r"\bpool\b",
        "laundry": r"laundry|washer|dryer|wash", "usage history": r"usage|consum|energy bills",
    }
    asked = [t for t, p in topics.items() if re.search(p, q)]
    hit = [t for t in asked if re.search(topics[t], low)]
    score = int(len(hit) / len(asked) * 100) if asked else 60
    fb = [f"Addresses {len(hit)}/{len(asked)} topics in the question" + (f" ({', '.join(hit)})" if hit else "")]
    if len(hit) < len(asked):
        fb.append(f"✗ Does not mention: {', '.join(t for t in asked if t not in hit)}")
    if "when" in q or "what time" in q or "best time" in q:
        if has_times:
            fb.append("✓ Gives the timing that was asked for")
        else:
            score = max(0, score - 20); fb.append("✗ Timing question but no times given")
    if "how much" in q or "save" in q or "cost" in q:
        if has_costs:
            fb.append("✓ Gives dollar figures as asked")
        else:
            score = max(0, score - 20); fb.append("✗ Cost question but no dollar figures given")
    result["relevance"] = {"score": score, "feedback": " | ".join(fb)}
 
    # ---------------- COMPLETENESS ----------------
    stop = {"should", "response", "with", "that", "from", "have", "this", "their", "which", "when", "using", "based"}
    terms = {w for w in re.findall(r"[a-z]{5,}", expected_response.lower()) if w not in stop}
    covered = {w for w in terms if w[:5] in low}
    missing = sorted(terms - covered)
    score = int((len(covered) / len(terms) if terms else 1) * 50)
    fb = [f"Covers {len(covered)}/{len(terms)} key terms of the expected response"]
    if missing:
        fb.append("✗ Not covered: " + ", ".join(missing[:6]))
    if any(w in low for w in ["recommend", "suggest", "should", "best", "optimal"]):
        score += 20; fb.append("✓ Gives a clear recommendation")
    else:
        fb.append("✗ No clear recommendation")
    if len(text.split()) >= 80:
        score += 15; fb.append("✓ Sufficient detail")
    else:
        fb.append("✗ Too brief")
    if re.search(r"^\s*(\d+\.|[-*•])\s", text, re.M):
        score += 15; fb.append("✓ Structured with steps/bullets")
    else:
        fb.append("✗ No structure (steps/bullets)")
    result["completeness"] = {"score": min(score, 100), "feedback": " | ".join(fb)}
 
    # ---------------- USEFULNESS ----------------
    score, fb = 0, []
    if any(v in low for v in ["set ", "run ", "charge", "schedule", "adjust", "turn off", "shift", "start "]):
        score += 30; fb.append("✓ Contains actionable advice")
    else:
        fb.append("✗ No clear action for the user")
    if has_times or has_temps:
        score += 30; fb.append("✓ Specific time/temperature the user can apply")
    else:
        fb.append("✗ Too vague to act on")
    if any(w in low for w in ["however", "but ", "consider", "alternatively", "if you"]):
        score += 20; fb.append("✓ Mentions trade-offs or alternatives")
    else:
        fb.append("○ Could mention trade-offs or alternatives")
    if has_costs or has_kwh:
        score += 20; fb.append("✓ Quantifies the benefit")
    else:
        fb.append("✗ Benefit not quantified")
    result["usefulness"] = {"score": score, "feedback": " | ".join(fb)}
 
    # ---------------- OVERALL ----------------
    overall = sum(result[k]["score"] for k in ("accuracy", "relevance", "completeness", "usefulness")) / 4
    weakest = min(("accuracy", "relevance", "completeness", "usefulness"), key=lambda k: result[k]["score"])
    result["overall"] = {
        "score": round(overall, 1),
        "grade": grade(overall),
        "feedback": f"Overall {round(overall, 1)}/100 ({grade(overall)}). Weakest area: {weakest} ({result[weakest]['score']}).",
    }
    return result

In [18]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used.
    Metrics: Tool Appropriateness, Tool Completeness (+ efficiency) with feedback for each."""
    import json
    from collections import Counter
 
    calls, sigs, errors, empty = [], Counter(), [], []
    for msg in messages:
        for tc in getattr(msg, "tool_calls", None) or []:           # tools the LLM requested
            calls.append(tc["name"])
            sigs[(tc["name"], json.dumps(tc.get("args", {}), sort_keys=True, default=str))] += 1
        if getattr(msg, "type", "") == "tool":
            content = str(msg.content)
            if content.lower().startswith(("error", "failed")):
                errors.append(msg.name)
            elif '"total_records": 0' in content:
                empty.append(msg.name)
 
    used = list(dict.fromkeys(calls))
    matched = [t for t in used if t in expected_tools]
    missing = [t for t in expected_tools if t not in used]
    extra = [t for t in used if t not in expected_tools]
 
    def grade(s):
        return "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 70 else "D" if s >= 60 else "F"
 
    why = {
        "get_weather_forecast": "needed for solar/weather-dependent decisions",
        "get_electricity_prices": "needed for cost optimisation",
        "query_energy_usage": "needed for personalised, history-based advice",
        "get_recent_energy_summary": "needed for a quick view of recent usage",
        "query_solar_generation": "needed to know the household's solar production",
        "calculate_energy_savings": "needed for exact savings figures instead of estimates",
        "search_energy_tips": "knowledge-base tips would back up the recommendations",
    }
 
    # ---------------- Tool Appropriateness (were the RIGHT tools selected?) ----------------
    app = int(len(matched) / len(used) * 100) if used else 0
    fb = []
    if matched: fb.append(f"✓ Appropriate tools: {', '.join(matched)}")
    if extra: fb.append(f"⚠ Not expected for this task: {', '.join(extra)} (may still be reasonable)")
    if not used: fb.append("✗ No tools were used - answer is not data-driven")
    tool_appropriateness = {"score": app, "feedback": " | ".join(fb)}
 
    # ---------------- Tool Completeness (were ALL necessary tools used?) ----------------
    comp = int((len(expected_tools) - len(missing)) / len(expected_tools) * 100) if expected_tools else 100
    fb = [f"Used {len(expected_tools) - len(missing)}/{len(expected_tools)} expected tools"]
    for t in missing:
        fb.append(f"✗ Missing {t} - {why.get(t, 'expected for this scenario')}")
    if not missing and expected_tools:
        fb.append("✓ All necessary tools were used")
    tool_completeness = {"score": comp, "feedback": " | ".join(fb)}
 
    # ---------------- Tool Efficiency (extra metric) ----------------
    dupes = {k[0]: c for k, c in sigs.items() if c > 1}
    eff = max(0, 100 - 15 * sum(c - 1 for c in dupes.values()) - 15 * len(errors))
    fb = []
    if dupes: fb.append(f"⚠ Identical calls repeated: {dupes}")
    if errors: fb.append(f"✗ Tool errors: {errors}")
    if empty: fb.append(f"○ Returned 0 records: {sorted(set(empty))} - check the date range")
    tool_efficiency = {"score": eff, "feedback": " | ".join(fb) or "✓ No redundant or failing calls"}
 
    overall = app * 0.3 + comp * 0.5 + eff * 0.2
    return {
        "used_tools": used,
        "expected_tools": expected_tools,
        "missing_tools": missing,
        "extra_tools": extra,
        "tool_appropriateness": tool_appropriateness,
        "tool_completeness": tool_completeness,
        "tool_efficiency": tool_efficiency,
        "overall": {
            "score": round(overall, 1),
            "grade": grade(overall),
            "feedback": f"Used {len(used)} tool(s); {len(matched)} expected, {len(missing)} missing, {len(extra)} unexpected.",
        },
    }

In [19]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results=None):
    """Generate a comprehensive evaluation report: overall scores, strengths, weaknesses, recommendations.
    Defaults to the notebook's `test_results`. Show it with display_evaluation_report(report)."""
    from collections import Counter
    from datetime import datetime
 
    if results is None:
        results = test_results
 
    details, errored = [], []
    for r in results:
        resp = r["response"]
        if not isinstance(resp, dict) or "messages" not in resp:
            errored.append(r["test_id"])
            continue
        msgs = resp["messages"]
        final = ""
        for m in reversed(msgs):                      # last AI message with text = final answer
            if getattr(m, "type", "") == "ai" and isinstance(m.content, str) and m.content.strip():
                final = m.content
                break
        details.append({
            "test_id": r["test_id"],
            "question": r["question"],
            "response_evaluation": evaluate_response(r["question"], final, r["expected_response"]),
            "tool_evaluation": evaluate_tool_usage(msgs, r["expected_tools"]),
        })
 
    n = len(details)
    avg = lambda vals: round(sum(vals) / len(vals), 1) if vals else 0.0
    rq = {k: avg([d["response_evaluation"][k]["score"] for d in details])
          for k in ("accuracy", "relevance", "completeness", "usefulness", "overall")}
    tu = {k: avg([d["tool_evaluation"][k]["score"] for d in details])
          for k in ("tool_appropriateness", "tool_completeness", "tool_efficiency", "overall")}
    passed = [d["test_id"] for d in details
              if d["response_evaluation"]["overall"]["score"] >= 70 and d["tool_evaluation"]["overall"]["score"] >= 70]
    missed = Counter(t for d in details for t in d["tool_evaluation"]["missing_tools"])
 
    strengths, weaknesses, recs = [], [], []
    for k in ("accuracy", "relevance", "completeness", "usefulness"):
        if rq[k] >= 80: strengths.append(f"{k.title()} is strong (avg {rq[k]}/100)")
        elif rq[k] < 70: weaknesses.append(f"{k.title()} is weak (avg {rq[k]}/100)")
    for k in ("tool_appropriateness", "tool_completeness", "tool_efficiency"):
        label = k.replace("_", " ").title()
        if tu[k] >= 80: strengths.append(f"{label} is strong (avg {tu[k]}/100)")
        elif tu[k] < 70: weaknesses.append(f"{label} is weak (avg {tu[k]}/100)")
 
    for tool, c in missed.most_common(3):
        recs.append(f"'{tool}' was skipped in {c}/{n} tests - make the system prompt require it for those question types")
    if rq["accuracy"] < 80:
        recs.append("Make the agent quote exact tool figures and show calculations (use calculate_energy_savings)")
    if rq["completeness"] < 80:
        recs.append("Enforce an answer format: direct answer, why (data), savings, tips with sources")
    if rq["relevance"] < 80:
        recs.append("Tell the agent to address every device/topic named in the question")
    if any("0 records" in d["tool_evaluation"]["tool_efficiency"]["feedback"] for d in details):
        recs.append("Database queries returned 0 records - pass dates relative to today and check the database is populated")
    if not recs:
        recs.append("Add harder edge cases (missing data, tool errors, off-topic questions)")
 
    return {
        "summary": {
            "total_tests": len(results),
            "evaluated_tests": n,
            "errored_tests": errored,
            "passed_tests": len(passed),
            "pass_rate": round(len(passed) / n * 100, 1) if n else 0.0,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        },
        "response_quality": rq,
        "tool_usage": tu,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recs,
        "passed": passed,
        "test_details": details,
    }
 

In [20]:
def display_evaluation_report(report, show_feedback=True):
    """Display the evaluation report in a readable format."""
    grade = lambda s: "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 70 else "D" if s >= 60 else "F"
    line = "=" * 80
    s = report["summary"]
 
    print(line + "\nECOHOME ENERGY ADVISOR - EVALUATION REPORT\n" + line)
    print(f"Tests: {s['total_tests']} | Evaluated: {s['evaluated_tests']} | "
          f"Passed: {s['passed_tests']} ({s['pass_rate']}%) | Errored: {s['errored_tests'] or 'none'}")
    print(f"Generated: {s['timestamp']}   (pass = response >= 70 and tools >= 70)")
 
    for title, block in (("RESPONSE QUALITY (0-100)", report["response_quality"]), ("TOOL USAGE (0-100)", report["tool_usage"])):
        print(f"\n{title}\n" + "-" * 80)
        for k, v in block.items():
            print(f"  {k.replace('_', ' ').title():<24}{v:>6}  ({grade(v)})")
 
    for title, key in (("STRENGTHS", "strengths"), ("WEAKNESSES", "weaknesses"), ("RECOMMENDATIONS", "recommendations")):
        print(f"\n{title}\n" + "-" * 80)
        for item in report[key] or ["None"]:
            print(f"  - {item}")
 
    print("\nPER-TEST RESULTS\n" + "-" * 80)
    for d in report["test_details"]:
        r, t = d["response_evaluation"], d["tool_evaluation"]
        status = "PASS" if d["test_id"] in report["passed"] else "FAIL"
        print(f"\n[{status}] {d['test_id']}  -  {d['question']}")
        print(f"   Response: {r['overall']['score']} ({r['overall']['grade']}) | Tools: {t['overall']['score']} ({t['overall']['grade']})")
        print(f"   Tools used: {', '.join(t['used_tools']) or 'none'}")
        if show_feedback:
            for k in ("accuracy", "relevance", "completeness", "usefulness"):
                print(f"   {k.title():<13}{r[k]['score']:>4}: {r[k]['feedback']}")
            print(f"   {'Tool approp.':<13}{t['tool_appropriateness']['score']:>4}: {t['tool_appropriateness']['feedback']}")
            print(f"   {'Tool compl.':<13}{t['tool_completeness']['score']:>4}: {t['tool_completeness']['feedback']}")
            print(f"   {'Tool effic.':<13}{t['tool_efficiency']['score']:>4}: {t['tool_efficiency']['feedback']}")
    print("\n" + line)
    print(f"Overall response score: {report['response_quality']['overall']}/100 | "
          f"Overall tool score: {report['tool_usage']['overall']}/100")
    print(line)

In [21]:
evaluation_report = generate_evaluation_report()
display_evaluation_report(evaluation_report)

ECOHOME ENERGY ADVISOR - EVALUATION REPORT
Tests: 10 | Evaluated: 10 | Passed: 5 (50.0%) | Errored: none
Generated: 2026-10-30T09:35:40   (pass = response >= 70 and tools >= 70)

RESPONSE QUALITY (0-100)
--------------------------------------------------------------------------------
  Accuracy                  87.0  (B)
  Relevance                 92.0  (A)
  Completeness              63.9  (D)
  Usefulness                85.0  (B)
  Overall                   82.0  (B)

TOOL USAGE (0-100)
--------------------------------------------------------------------------------
  Tool Appropriateness      71.6  (C)
  Tool Completeness         61.5  (D)
  Tool Efficiency          100.0  (A)
  Overall                   72.2  (C)

STRENGTHS
--------------------------------------------------------------------------------
  - Accuracy is strong (avg 87.0/100)
  - Relevance is strong (avg 92.0/100)
  - Usefulness is strong (avg 85.0/100)
  - Tool Efficiency is strong (avg 100.0/100)

WEAKNESSES
-----